# 04_extract_text

Dry-run text extraction on the latest inventory output. This notebook stays conservative: text-like files, PDF text extraction, DOCX paragraphs/tables, and XLSX sheet previews. No OCR yet.


In [2]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.inventory import ensure_inventory_schema
from src.extractors import ExtractConfig, enrich_inventory_with_text, save_text_outputs

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
inventory_files = [p for p in OUTPUT_DIR.glob('inventory_*.parquet') if not p.name.startswith('inventory_with_text_')]
assert inventory_files, 'No inventory parquet files found. Run 02_inventory.ipynb first.'
# Pick the newest file by filesystem timestamp, not filename order.
INVENTORY_PATH = max(inventory_files, key=lambda p: p.stat().st_mtime)
print('Using inventory file:', INVENTORY_PATH.name)


Using inventory file: inventory_HTL0049-01_OITYLO-KOKKALA_MANI_20260309_133149.parquet


In [3]:
inv = pd.read_parquet(INVENTORY_PATH)
inv = ensure_inventory_schema(inv)
print('Rows:', len(inv))
preview_cols = [c for c in ['relative_path', 'suffix', 'size_bytes'] if c in inv.columns]
display(inv[preview_cols].head(10))
if 'suffix' in inv.columns:
    display(inv['suffix'].fillna('').value_counts().rename_axis('suffix').reset_index(name='count').head(20))
else:
    print('suffix column unavailable after schema backfill')


Rows: 4241


,relative_path,suffix,size_bytes
0,.DS_Store,,14340
1,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\.DS_Store,,10244
2,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,.pdf,152816
3,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,.pdf,124890
4,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,.pdf,67290
5,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΕΓΚΡΙΣΗ ΕΦΟΡΕΙΑΣ ΑΡΧΑ...,.pdf,564756
6,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΕΛΕΓΧΟΣ ΔΟΜΗΣΗΣ\ΠΟΡΙΣ...,.pdf,415711
7,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\01_SONADO IK...,.pdf,676187
8,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\02_SONADO IK...,.pdf,2386336
9,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\03_SONADO IK...,.pdf,12993826


,suffix,count
0,.pdf,2403
1,.jpeg,664
2,.jpg,355
3,.dwg,253
4,.msg,121
5,.bak,78
6,.docx,68
7,,64
8,.db,50
9,.xlsx,49


In [4]:
config = ExtractConfig(
    max_chars_per_file=12000,
    max_csv_rows=30,
    max_csv_columns=20,
    max_xlsx_rows_per_sheet=30,
    max_xlsx_columns=20,
    max_docx_paragraphs=300,
    max_pdf_pages=30,
    preview_chars=300,
)
config


ExtractConfig(max_chars_per_file=12000, max_csv_rows=30, max_csv_columns=20, max_xlsx_rows_per_sheet=30, max_xlsx_columns=20, max_docx_paragraphs=300, max_pdf_pages=30, preview_chars=300)

In [7]:
enriched = enrich_inventory_with_text(inv, path_column='absolute_path', config=config)
display(enriched[['relative_path', 'suffix', 'text_status', 'text_source', 'extracted_chars', 'text_preview']].head(20))


incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
Multiple definitions in dictionary at byte 0x375e3 for key /Info
Multiple definitions in dictionary at byte 0x375ef for key /Info
Multiple definitions in dictionary at byte 0x375fb for key /Info
Multiple definitions in dictionary at byte 0x4c8e0 for key /Info
Multiple definitions in dictionary at byte 0x4c8ec for key /Info
Multiple definitions in dictionary at byte 0x4c8f8 for key /Info
Multiple definitions in dictionary at byte 0x44fac for key /Info
Multiple definitions in dictionary at byte 0x44fb8 for key /Info
Multiple definitions in dictionary at byte 0x44fc4 for key /Info
Multiple definitions in dictionary at byte 0x44fac for key /Info
Multiple definitions in dictionary at byte 0x44fb8 for key /Info
Multiple definitions in dictionary at byte 0x44fc4 for key /Info
Multiple definitions in dictionary at byte 0x3f676 for key /Info
Multiple definitions in dictionary at b

,relative_path,suffix,text_status,text_source,extracted_chars,text_preview
0,.DS_Store,,unsupported,unsupported,0,
1,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\.DS_Store,,unsupported,unsupported,0,
2,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,.pdf,ok,pdf,3412,Δ Η Λ Ω Σ Η Α Ν Α Θ Ε Σ Ε Ω Ν ΕΡΓΟ : ΑΝΕΓΕΡΣΗ ...
3,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,.pdf,ok,pdf,1206,ΔΗΛΩΣΕΙΣ ΑΝΑΛΗΨΗΣ ΜΕΛΕΤΩΝ/ΕΠΙΒΛΕΨΕΩΝ ΕΡΓΟ : ΑΝ...
4,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,.pdf,empty,pdf,0,
5,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΕΓΚΡΙΣΗ ΕΦΟΡΕΙΑΣ ΑΡΧΑ...,.pdf,ok,pdf,6365,1 E Λ Λ Η Ν Ι Κ Η Δ Η Μ Ο Κ Ρ Α T Ι Α ΥΠΟΥΡΓΕΙ...
6,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΕΛΕΓΧΟΣ ΔΟΜΗΣΗΣ\ΠΟΡΙΣ...,.pdf,ok,pdf,1937,ΠΟΡΙΣΜΑ ΕΛΕΓΚΤΩΝ ΔΟΜΗΣΗΣ Αρ. πρωτ. Ορισμού: 15...
7,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\01_SONADO IK...,.pdf,ok,pdf,12014,ΕΡΓΟΔΟΤΗΣ: SONADO ΜΟΝΟΠΡΟΣΩΠΗ Ι.Κ.Ε. ΕΡΓΟ: ΑΝΕ...
8,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\02_SONADO IK...,.pdf,ok,pdf,8560,"E (1,2,3,4,5,6,7,8,9,10,11,12,13,.......,62,63..."
9,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\03_SONADO IK...,.pdf,ok,pdf,12015,ΜΑΡΤΙΟΣ 2021 ΜΕΛΕΤΗΤΕΣ Π.Ε ΟΔΟΣ - ΘΕΣΗ ΔΗΜΟΣ Ε...


In [13]:
display(enriched[enriched['text_status'] == 'error'][['relative_path', 'suffix', 'text_error']])

,relative_path,suffix,text_error
4054,ΦΑΚΕΛΟΣ ΑΝΑΠΤΥΞΙΑΚΟΥ ΝΟΜΟΥ\ΕΓΚΡΙΣΕΙΣ - ΑΠΟΦΑΣΕ...,.pdf,PdfStreamError: Stream has ended unexpectedly


In [14]:
display(enriched['text_status'].value_counts(dropna=False).rename_axis('text_status').reset_index(name='count'))
display(enriched['text_source'].value_counts(dropna=False).rename_axis('text_source').reset_index(name='count'))
display(enriched[enriched['text_status'] == 'error'][['relative_path', 'suffix', 'text_error']].head(20))


,text_status,count
0,ok,1926
1,unsupported,1677
2,empty,637
3,error,1


,text_source,count
0,pdf,2403
1,unsupported,1677
2,docx,68
3,xlsx,49
4,text,44


,relative_path,suffix,text_error
4054,ΦΑΚΕΛΟΣ ΑΝΑΠΤΥΞΙΑΚΟΥ ΝΟΜΟΥ\ΕΓΚΡΙΣΕΙΣ - ΑΠΟΦΑΣΕ...,.pdf,PdfStreamError: Stream has ended unexpectedly


In [15]:
display(enriched[enriched['has_extracted_text']][['relative_path', 'text_source', 'extracted_chars', 'text_preview']].head(20))


,relative_path,text_source,extracted_chars,text_preview
2,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,pdf,3412,Δ Η Λ Ω Σ Η Α Ν Α Θ Ε Σ Ε Ω Ν ΕΡΓΟ : ΑΝΕΓΕΡΣΗ ...
3,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,pdf,1206,ΔΗΛΩΣΕΙΣ ΑΝΑΛΗΨΗΣ ΜΕΛΕΤΩΝ/ΕΠΙΒΛΕΨΕΩΝ ΕΡΓΟ : ΑΝ...
5,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΕΓΚΡΙΣΗ ΕΦΟΡΕΙΑΣ ΑΡΧΑ...,pdf,6365,1 E Λ Λ Η Ν Ι Κ Η Δ Η Μ Ο Κ Ρ Α T Ι Α ΥΠΟΥΡΓΕΙ...
6,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΕΛΕΓΧΟΣ ΔΟΜΗΣΗΣ\ΠΟΡΙΣ...,pdf,1937,ΠΟΡΙΣΜΑ ΕΛΕΓΚΤΩΝ ΔΟΜΗΣΗΣ Αρ. πρωτ. Ορισμού: 15...
7,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\01_SONADO IK...,pdf,12014,ΕΡΓΟΔΟΤΗΣ: SONADO ΜΟΝΟΠΡΟΣΩΠΗ Ι.Κ.Ε. ΕΡΓΟ: ΑΝΕ...
8,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\02_SONADO IK...,pdf,8560,"E (1,2,3,4,5,6,7,8,9,10,11,12,13,.......,62,63..."
9,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\03_SONADO IK...,pdf,12015,ΜΑΡΤΙΟΣ 2021 ΜΕΛΕΤΗΤΕΣ Π.Ε ΟΔΟΣ - ΘΕΣΗ ΔΗΜΟΣ Ε...
10,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\04_SONADO IK...,pdf,1787,"SONADO ΜΟΝΟΠΡΟΣΩΠΗ Ι.Κ.Ε. ΘΕΣΗ ""ΚΟΚΚΑΛΑ"" (ΕΚΤΟ..."
11,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\05_SONADO IK...,pdf,3315,"SONADO ΜΟΝΟΠΡΟΣΩΠΗ Ι.Κ.Ε. ΘΕΣΗ ""ΚΟΚΚΑΛΑ"" (ΕΚΤΟ..."
12,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\06_SONADO IK...,pdf,3248,"SONADO ΜΟΝΟΠΡΟΣΩΠΗ Ι.Κ.Ε. ΘΕΣΗ ""ΚΟΚΚΑΛΑ"" (ΕΚΤΟ..."


In [16]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_base = OUTPUT_DIR / f'inventory_with_text_{timestamp}'
csv_path, parquet_path = save_text_outputs(enriched, output_base)
print('Saved CSV   :', csv_path)
print('Saved Parquet:', parquet_path)


Saved CSV   : c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\inventory_with_text_20260309_154442.csv
Saved Parquet: c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\inventory_with_text_20260309_154442.parquet


In [17]:
err = enriched[enriched['text_status'] == 'error'].copy()
long_err = err[err['path_length'] > 250].copy()
fnf_long = long_err[long_err['text_error'].fillna('').str.contains('FileNotFoundError', regex=False)]
print('errors_total =', len(err))
print('errors_path_gt_250 =', len(long_err))
print('FileNotFoundError_on_path_gt_250 =', len(fnf_long))
display(long_err[['relative_path', 'path_length', 'suffix', 'text_error']].head(20))

errors_total = 1
errors_path_gt_250 = 0
FileNotFoundError_on_path_gt_250 = 0


,relative_path,path_length,suffix,text_error


In [6]:
from pathlib import Path
import pandas as pd
from src.executor import copy_and_rename_from_paths

xlsx_path = Path(r"C:\SONADO-IK-801455240\inventory_with_target_absolute_path_and_reason_20260310_110815.xlsx")
if not xlsx_path.exists():
    raise FileNotFoundError(f"Missing Excel file: {xlsx_path}")

manifest_df = pd.read_excel(xlsx_path)
print(f"Loaded {len(manifest_df):,} rows from {xlsx_path.name}")
print("Available columns:")
for col in manifest_df.columns:
    print(f" - {col}")

def ask_column_name(label: str) -> str:
    while True:
        selected = input(f"Enter the column name for {label}: ").strip()
        if selected in manifest_df.columns:
            return selected
        print(f"Column '{selected}' not found. Please choose one of the listed columns.")

source_col = ask_column_name("source absolute path")
target_col = ask_column_name("target absolute path")

copy_manifest = manifest_df.copy()
copy_manifest["absolute_current_path"] = copy_manifest[source_col]
copy_manifest["absolute_proposed_path"] = copy_manifest[target_col]

log_df = copy_and_rename_from_paths(copy_manifest, overwrite_existing=True)
log_df[["copy_status", "copy_message"]].value_counts(dropna=False).rename("rows")

Loaded 4,241 rows from inventory_with_target_absolute_path_and_reason_20260310_110815.xlsx
Available columns:
 - scan_root
 - absolute_path
 - relative_path
 - parent_relative
 - filename
 - stem
 - suffix
 - modified_at
 - created_at
 - depth_segments
 - path_length
 - hash
 - is_duplicate_hash
 - extracted_text
 - text_preview
 - has_extracted_text
 - target_absolute_path
 - move_reason_category


copy_status     copy_message                  
error           missing absolute_proposed_path    2238
copied          copy and rename completed         1902
missing_source  source file does not exist         101
Name: rows, dtype: int64

In [1]:
import pandas as pd

csv_path = r"c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\inventory_with_text_20260309_154442.csv"
df = pd.read_csv(csv_path)

df.head()

,scan_root,absolute_path,relative_path,parent_relative,filename,stem,suffix,size_bytes,modified_at,created_at,...,text_status,text_source,extracted_text,text_preview,extracted_chars,text_truncated,text_error,extracted_pages,extracted_sheets,has_extracted_text
0,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,.DS_Store,NaN,.DS_Store,.DS_Store,NaN,14340,2026-02-27 09:59:43.850801229,2026-03-09 09:23:17.232372999,...,unsupported,unsupported,NaN,NaN,0,False,NaN,NaN,NaN,False
1,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\.DS_Store,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ,.DS_Store,.DS_Store,NaN,10244,2025-12-08 08:24:58.179627657,2026-03-09 09:23:18.215867043,...,unsupported,unsupported,NaN,NaN,0,False,NaN,NaN,NaN,False
2,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,01_SONADO_OITYLO_DHLWSH ANATHESHS.pdf,01_SONADO_OITYLO_DHLWSH ANATHESHS,.pdf,152816,2022-04-15 13:51:42.468893051,2026-03-09 09:23:18.831106901,...,ok,pdf,Δ Η Λ Ω Σ Η Α Ν Α Θ Ε Σ Ε Ω Ν\nΕΡΓΟ : ΑΝΕΓ...,Δ Η Λ Ω Σ Η Α Ν Α Θ Ε Σ Ε Ω Ν ΕΡΓΟ : ΑΝΕΓΕΡΣΗ ...,3412,False,NaN,1.0,NaN,True
3,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,02_SONADO_OITYLO_DHLWSH ANALHPSHS_STFNLAB OE.pdf,02_SONADO_OITYLO_DHLWSH ANALHPSHS_STFNLAB OE,.pdf,124890,2022-04-15 13:51:43.421580553,2026-03-09 09:23:19.272340298,...,ok,pdf,ΔΗΛΩΣΕΙΣ ΑΝΑΛΗΨΗΣ ΜΕΛΕΤΩΝ/ΕΠΙΒΛΕΨΕΩΝ\nΕΡΓΟ : ...,ΔΗΛΩΣΕΙΣ ΑΝΑΛΗΨΗΣ ΜΕΛΕΤΩΝ/ΕΠΙΒΛΕΨΕΩΝ ΕΡΓΟ : ΑΝ...,1206,False,NaN,1.0,NaN,True
4,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,C:\SONADO-IK-801455240\HTL0049-01_OITYLO-KOKKA...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,03_SONADO_OITYLO_DHLWSH ANALHPSHS_GKA ENGINEER...,03_SONADO_OITYLO_DHLWSH ANALHPSHS_GKA ENGINEERS,.pdf,67290,2022-04-15 13:51:44.311928749,2026-03-09 09:23:19.710536003,...,empty,pdf,NaN,NaN,0,False,NaN,1.0,NaN,False


In [17]:
import re

# Excel/OpenXML disallows most control chars; keep tab/newline/carriage return.
illegal_ctrl = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")


def sanitize_for_excel(value):
    if isinstance(value, str):
        return illegal_ctrl.sub(" ", value)
    return value


df_export = df.copy()
text_cols = df_export.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df_export[col] = df_export[col].map(sanitize_for_excel)

cols = [
    "scan_root", "absolute_path", "relative_path", "parent_relative",
    "filename", "stem", "suffix", "modified_at", "created_at",
    "depth_segments", "path_length", "hash", "is_duplicate_hash", "extracted_text",
    "text_preview", "has_extracted_text"
]

output_csv = "inventory_with_text_reduced_20260309_154442.csv"
df_export.to_csv(output_csv, columns=cols, index=False)
print(f"Saved: {output_csv}")

Saved: inventory_with_text_reduced_20260309_154442.csv
